# Session 9 · Advanced Prompting Strategies and AI in Spreadsheets

Generative AI › Track 1 · Prompt Engineering

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Runtime ▸ Run all in Colab). Outputs below were produced by actually executing this notebook.

### Example · `session-09_prompt_linter.py`

In [1]:
"""A prompt 'linter': checks a prompt against expert strategies and suggests fixes."""
import re
CHECKS = [
    ("Has an action verb", lambda p: re.match(r"^(write|explain|list|compare|summari[sz]e|create|draft|classify|translate|generate|analy[sz]e|rewrite|act)\b", p.strip(), re.I) or re.search(r"\byou are\b", p, re.I),
     "Start with a clear verb (Write, Explain, Compare...) or a role line."),
    ("States audience", lambda p: re.search(r"\b(for|audience|students|beginners|manager|customers|kids)\b", p, re.I),
     "Say who the output is for."),
    ("Limits length", lambda p: re.search(r"\b(\d+\s*(words|bullets|sentences|lines|characters)|short|brief)\b", p, re.I),
     "Add a length limit (e.g. 'under 120 words')."),
    ("Specifies format", lambda p: re.search(r"\b(table|bullet|json|list|steps|headings|markdown|csv)\b", p, re.I),
     "Ask for a format: table, bullets, JSON..."),
    ("Uses delimiters for input", lambda p: ('"""' in p) or ("###" in p) or ("<text>" in p) or ("Text:" not in p),
     "Wrap pasted input in delimiters like ### or triple quotes."),
    ("Avoids vague words", lambda p: not re.search(r"\b(good|nice|stuff|things|etc)\b", p, re.I),
     "Replace vague words with measurable criteria."),
]

def lint(prompt):
    passed, tips = 0, []
    for name, test, tip in CHECKS:
        if test(prompt): passed += 1
        else: tips.append(f"  - {name}: {tip}")
    return passed, tips

for p in ["Write something good about our product etc.",
          "You are a product marketer. Write 3 bullet points (under 15 words each) for busy managers "
          "explaining why our attendance app saves time. Format: Markdown list."]:
    score, tips = lint(p)
    print(f"PROMPT: {p[:70]}...\nScore: {score}/{len(CHECKS)}")
    print("\n".join(tips) if tips else "  All checks passed")
    print()

PROMPT: Write something good about our product etc....
Score: 2/6
  - States audience: Say who the output is for.
  - Limits length: Add a length limit (e.g. 'under 120 words').
  - Specifies format: Ask for a format: table, bullets, JSON...
  - Avoids vague words: Replace vague words with measurable criteria.

PROMPT: You are a product marketer. Write 3 bullet points (under 15 words each...
Score: 6/6
  All checks passed



### Example · `session-09_sheet_ai_formula.py`

In [2]:
"""Simulates an '=AI(prompt, cell)' column in a spreadsheet: one prompt template applied per row.
The model call is replaced by a deterministic rule so the demo runs offline (SAMPLE behaviour)."""
rows = [("Order #101", "The parcel arrived broken and support never replied."),
        ("Order #102", "Fast delivery, great packaging, will buy again!"),
        ("Order #103", "Item is okay, but delivery took 9 days.")]
TEMPLATE = "Classify the customer feedback as Positive, Negative or Mixed and give a 5-word reason: {text}"

def fake_model(prompt):
    t = prompt.lower()
    neg = sum(w in t for w in ["broken", "never", "took 9 days", "late"])
    pos = sum(w in t for w in ["fast", "great", "again", "okay"])
    label = "Mixed" if neg and pos else ("Negative" if neg else "Positive")
    reason = {"Negative": "damaged item, no support reply", "Positive": "fast delivery and good packaging",
              "Mixed": "fine product, slow delivery time"}[label]
    return label, reason

print(f"{'Row':11} | {'Label':8} | Reason")
for order, text in rows:
    label, reason = fake_model(TEMPLATE.format(text=text))
    print(f"{order:11} | {label:8} | {reason}")
print("\nPrompt sent per row:\n" + TEMPLATE.format(text=rows[0][1]))

Row         | Label    | Reason
Order #101  | Negative | damaged item, no support reply
Order #102  | Positive | fast delivery and good packaging
Order #103  | Mixed    | fine product, slow delivery time

Prompt sent per row:
Classify the customer feedback as Positive, Negative or Mixed and give a 5-word reason: The parcel arrived broken and support never replied.
